# LRP-Merge: Layer-wise Relevance Propagation for Model Merging

This notebook demonstrates the LRP-Merge method, which uses Layer-wise Relevance Propagation to identify and preserve the most important weights from multiple task-specific models during a merge.

### Workflow:
1. **Setup**: Mount Drive and install dependencies.
2. **Training**: Fine-tune LoRA adapters for specific tasks.
3. **LRP Analysis**: Compute relevance scores for model weights.
4. **Model Preparation**: Reconstruct full models from adapters.
5. **Merging**: Perform LRP-Merge using a custom Mergekit implementation.
6. **Evaluation**: Test the merged model and optimize merge density.

### **Note**-
1. Install 'mergekit' inorder to execute this notebook.
2. The paths mentioned in the code depend on your folder structure in google drive(if you are using colab) or your local system.
3. Here, I have used "TinyLlama/TinyLlama-1.1B-Chat-v1.0" for this experiment only.
4. I had use FAKE_NEWS_DETECTION dataset to test the merged model only for this experiment.

## Download Base Model
Before merging, we need to download the base model weights from Hugging Face and store them in our project folder.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
LRP_PATH="your_folder_path"
# Define the base model ID and local storage path
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" #or use any other model you like
local_base_path = os.path.join(LRP_PATH, "base_model")

if not os.path.exists(local_base_path):
    print(f"Downloading {base_model_id}...")
    model = AutoModelForCausalLM.from_pretrained(base_model_id)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)

    os.makedirs(local_base_path, exist_ok=True)
    model.save_pretrained(local_base_path)
    tokenizer.save_pretrained(local_base_path)
    print(f"Base model saved to: {local_base_path}")
else:
    print(f"Base model already exists at: {local_base_path}")

In [ ]:
#command to install mergekit
!pip install -q -U mergekit

In [14]:
import os

# --- PORTABLE PATH SETUP ---
# If running in Colab with Drive: "/content/drive/MyDrive/LRP Merge method"
# If running locally or after cloning: "."
BASE_DIR = "."
os.chdir(BASE_DIR)

# Define relative subdirectories
MODEL_DIR = "models"
DATA_DIR = "datasets"
REPO_DIR = "mergekit_repo"

# Create directories if they don't exist
for d in [MODEL_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Project root set to: {os.getcwd()}")
print(f"Models will be saved to: {os.path.abspath(MODEL_DIR)}")

✅ Project root set to: /content
Models will be saved to: /content/models


In [19]:
import os

# Path to your project folder
LRP_PATH = "your_path"
gitignore_path = os.path.join(LRP_PATH, ".gitignore")

gitignore_content = """
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class

# Model weights and Large Files (Crucial for GitHub)
models/
base_model/
mergekit_repo/venv/
*.bin
*.safetensors
*.pt
*.pth
*.zip
*.tar.gz

# Colab/Notebook specific
.ipynb_checkpoints/
.virtual_documents/

# Environments
.env
.venv
env/
venv/

# OS generated files
.DS_Store
ehthumbs.db
Thumbs.db
"""

# Ensure the directory exists (it should, but safety first)
os.makedirs(LRP_PATH, exist_ok=True)

with open(gitignore_path, "w") as f:
    f.write(gitignore_content.strip())

print(f"✅ Created .gitignore at: {gitignore_path}")
print("This file will prevent large models and temporary files from being tracked by Git.")

✅ Created .gitignore at: /content/drive/MyDrive/LRP Merge method/.gitignore
This file will prevent large models and temporary files from being tracked by Git.


In [15]:
gitignore_content = """
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class

# Model weights (Crucial: do not push these!)
models/
base_model/
*.bin
*.safetensors
*.zip

# Environments
.env
.venv
env/
venv/

# Notebook checkpoints
.ipynb_checkpoints
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content.strip())

print("✅ Created .gitignore to protect your repo from large model files.")

✅ Created .gitignore to protect your repo from large model files.


In [16]:
import subprocess

def run_portable_merge(config_name, output_name):
    # Use relative paths for the merge command
    output_path = os.path.join(MODEL_DIR, output_name)
    os.makedirs(output_path, exist_ok=True)

    cmd = [
        "mergekit-yaml",
        config_name,
        output_path,
        "--copy-tokenizer",
        "--allow-crimes",
        "--lazy-unpickle"
    ]

    print(f"Executing portable merge: {' '.join(cmd)}")
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print(f"✅ Successfully merged to {output_path}")
    else:
        print(f"✗ Error: {res.stderr}")

In [13]:
import os

# Define path for this cell scope
LRP_PATH = "your_folder_path"

print("--- Files and Directories in LRP Project ---")
if os.path.exists(LRP_PATH):
    # List files and indicate if they are directories
    for item in os.listdir(LRP_PATH):
        full_path = os.path.join(LRP_PATH, item)
        if os.path.isdir(full_path):
            print(f"[DIR]  {item}/")
        else:
            print(f"[FILE] {item}")
else:
    print(f"Error: {LRP_PATH} not found.")

print("\n--- Inside 'models/' (Checking for weights) ---")
models_path = os.path.join(LRP_PATH, "models")
if os.path.exists(models_path):
    !ls -F "{models_path}"


--- Files and Directories in LRP Project ---
[FILE] finetune_fakenews.py
[FILE] lrp_computer.py
[FILE] INSTRUCTIONS_FOR_COLAB.md
[DIR]  datasets/
[DIR]  models/
[DIR]  __pycache__/
[FILE] lrp_config_colab.yaml
[DIR]  mergekit_repo/
[DIR]  base_model/
[FILE] lrp_merge_pipeline.py
[FILE] LRP_Merge_Colab_Training.ipynb

--- Inside 'models/' (Checking for weights) ---
compare_lrp/	merged-model/	    tinyllama-global-full/
compare_slerp/	merged-model-d0.5/  tinyllama-local/
compare_ties/	merged-model-d0.7/  tinyllama-local-full/
lrp-global/	merged-model-d0.9/
lrp-local/	tinyllama-global/


##Mount Google Drive

This connects your Google Drive to access the LRP merge method files.

In [8]:
# Step 1: Environment Setup
import os
from google.colab import drive

drive.mount('/content/drive') #should be used only once for colab
LRP_PATH = "your_folder_path"
os.chdir(LRP_PATH)

# Install standard dependencies
!pip install -q transformers datasets accelerate peft bitsandbytes safetensors numpy

# Install the custom LRP-enabled Mergekit from your Drive
mergekit_repo_path = os.path.join(LRP_PATH, "mergekit_repo")
!pip install -e "{mergekit_repo_path}"

print("\n✅ Environment ready and Custom Mergekit installed!")

Drive not mounted, so nothing to flush and unmount.
Successfully unmounted Google Drive.
Successfully removed directory: /content/drive
Mounted at /content/drive
Google Drive mounted successfully!


##  Installing Dependencies

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes safetensors
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pandas scikit-learn

print("\nAll dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00

All dependencies installed!


## Training Task-Specific Adapters
In this step, we fine-tune two LoRA adapters (Global and Local) on fake news detection datasets.

In [ ]:
# Verify datasets are available
import os
import pandas as pd

print("="*60)
print("Checking for datasets...")
print("="*60)

# Check for synthetic dataset
if os.path.exists("datasets/synthetic/train.csv"): #these paths depend on your folder structure
    df_train = pd.read_csv("datasets/synthetic/train.csv")
    df_test = pd.read_csv("datasets/synthetic/test.csv") if os.path.exists("datasets/synthetic/test.csv") else None

    print("\n✓ Synthetic dataset found:")
    print(f"  Train samples: {len(df_train)}")
    if df_test is not None:
        print(f"  Test samples: {len(df_test)}")
    print(f"  Columns: {list(df_train.columns)}")
    print(f"  Labels in train: {df_train['label'].value_counts().to_dict()}")

    # Show sample
    print("\n  Sample data:")
    print(df_train.head(3).to_string())
else:
    print("\n✗ Synthetic dataset not found!")
    print("  Expected: datasets/synthetic/train.csv")
    print("\nTo create sample data, run:")
    print("  !python download_fakenews_datasets.py --dataset synthetic --output ./datasets")
    print("\nOr create the folder structure manually with your CSV files.")

Checking for datasets...

✓ Synthetic dataset found:
  Train samples: 800
  Test samples: 200
  Columns: ['text', 'label']
  Labels in train: {'REAL': 404, 'FAKE': 396}

  Sample data:
                                                                      text label
0              Report shows economy has increased by 15% over past decade.  REAL
1  Government announces new policy regarding AI after public consultation.  REAL
2  Study finds correlation between AI and health benefits in new research.  REAL


## Step 8: Train GLOBAL Model

This trains the general knowledge model.

In [ ]:
# Configuration for GLOBAL model (using your existing dataset)
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  #Use larger model with GPU!
# MODEL_NAME = "gpt2"  # Or use smaller model if memory is limited

#Use your synthetic dataset
DATASET = "datasets/synthetic/train.csv"  # Your downloaded dataset
OUTPUT = "models/tinyllama-global"

# Training parameters
EPOCHS = 3
BATCH_SIZE = 4  # Can increase with GPU
MAX_SAMPLES = 1000  # Use all 800 samples or limit

print(f"Training GLOBAL model:")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {DATASET}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max samples: {MAX_SAMPLES}")
print(f"  Output: {OUTPUT}")

Training GLOBAL model:
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Dataset: datasets/synthetic/train.csv
  Epochs: 3
  Batch size: 4
  Max samples: 1000
  Output: models/tinyllama-global


In [ ]:
# (Assuming datasets are present in LRP_PATH/datasets)
# Train Global and Local models using the finetune_fakenews.py script
!python finetune_fakenews.py --dataset datasets/synthetic/train.csv --output models/tinyllama-global --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --epochs 3 --use-lora
!python finetune_fakenews.py --dataset datasets/synthetic/train.csv --output models/tinyllama-local --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --epochs 3 --use-lora

2026-03-29 07:38:50,334 - INFO - Starting fine-tuning...
2026-03-29 07:38:50,334 - INFO -   Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
2026-03-29 07:38:50,334 - INFO -   Dataset: datasets/synthetic/train.csv
2026-03-29 07:38:50,334 - INFO -   Output: models/tinyllama-global
2026-03-29 07:38:50,334 - INFO -   Epochs: 3
2026-03-29 07:38:50,334 - INFO -   LoRA: True
2026-03-29 07:38:50,334 - INFO -   8-bit: False
2026-03-29 07:38:50,334 - INFO -   4-bit: False
2026-03-29 07:38:50,334 - INFO - Loading dataset from datasets/synthetic/train.csv...
2026-03-29 07:38:50,340 - INFO - Loaded 800 samples
2026-03-29 07:38:50,340 - INFO -   FAKE: 396
2026-03-29 07:38:50,340 - INFO -   REAL: 404
2026-03-29 07:38:50,340 - INFO - Loading tokenizer and model...
2026-03-29 07:38:50,494 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 07:38:50,505 - INFO - HTTP Request: HEAD https://huggingface.co/api

##Train LOCAL Model

This trains the task-specific model.

In [ ]:
# Configuration for LOCAL model
# Using the same synthetic dataset for task-specific training
# In a real scenario, you might have a different dataset for the local model

LOCAL_OUTPUT = "models/tinyllama-local"
LOCAL_DATASET = "datasets/synthetic/train.csv"  # Can be same or different dataset

print(f"Training LOCAL model:")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {LOCAL_DATASET}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max samples: {MAX_SAMPLES}")
print(f"  Output: {LOCAL_OUTPUT}")
print(f"\nNote: Both models currently use the same dataset.")
print("For a true LRP-Merge experiment, train on different datasets:")
print("  - GLOBAL: General knowledge/tasks")
print("  - LOCAL: Specific task/domain")

Training LOCAL model:
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Dataset: datasets/synthetic/train.csv
  Epochs: 3
  Batch size: 4
  Max samples: 1000
  Output: models/tinyllama-local

Note: Both models currently use the same dataset.
For a true LRP-Merge experiment, train on different datasets:
  - GLOBAL: General knowledge/tasks
  - LOCAL: Specific task/domain


In [ ]:
!python finetune_fakenews.py \
    --dataset {LOCAL_DATASET} \
    --output {LOCAL_OUTPUT} \
    --model {MODEL_NAME} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --use-lora \
    --lora-r 16 \
    --max-samples {MAX_SAMPLES} \
    --max-length 256

2026-03-29 07:45:23,041 - INFO - Starting fine-tuning...
2026-03-29 07:45:23,041 - INFO -   Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
2026-03-29 07:45:23,041 - INFO -   Dataset: datasets/synthetic/train.csv
2026-03-29 07:45:23,041 - INFO -   Output: models/tinyllama-local
2026-03-29 07:45:23,041 - INFO -   Epochs: 3
2026-03-29 07:45:23,041 - INFO -   LoRA: True
2026-03-29 07:45:23,041 - INFO -   8-bit: False
2026-03-29 07:45:23,041 - INFO -   4-bit: False
2026-03-29 07:45:23,041 - INFO - Loading dataset from datasets/synthetic/train.csv...
2026-03-29 07:45:23,049 - INFO - Loaded 800 samples
2026-03-29 07:45:23,049 - INFO -   FAKE: 396
2026-03-29 07:45:23,049 - INFO -   REAL: 404
2026-03-29 07:45:23,049 - INFO - Loading tokenizer and model...
2026-03-29 07:45:23,409 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 07:45:23,429 - INFO - HTTP Request: HEAD https://huggingface.co/api/

## Computing LRP Scores
We calculate the importance of each weight using the LRP epsilon rule.

In [ ]:
!python lrp_merge_pipeline.py --compute-lrp --model models/tinyllama-global --output models/lrp-global
!python lrp_merge_pipeline.py --compute-lrp --model models/tinyllama-local --output models/lrp-local

Sun Mar 29 07:52:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Compute LRP for LOCAL model
!python lrp_merge_pipeline.py --compute-lrp \
    --model {LOCAL_OUTPUT} \
    --output models/lrp-local \
    --device cuda

print("\n✓ LRP scores computed for LOCAL model")

Sun Mar 29 07:53:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Reconstructing Full Models
LRP-Merge performs best when merging full model weights rather than raw adapters.

In [ ]:
# Create LRP config file with corrected structure
lrp_config = f"""merge_method: lrp

base_model:
  model: "{MODEL_NAME}"

parameters:
  density: 0.7
  use_lrp: true

models:
  - model:
      model: "{OUTPUT}"
    parameters:
      weight: 1.0
      lrp_scores:
        value: "./models/lrp-global/lrp_scores"
  - model:
      model: "{LOCAL_OUTPUT}"
    parameters:
      weight: 1.0
      lrp_scores:
        value: "./models/lrp-local/lrp_scores"
"""

with open("lrp_config_colab.yaml", "w") as f:
    f.write(lrp_config)

print("✓ Created lrp_config_colab.yaml with corrected structure")
!cat lrp_config_colab.yaml

✓ Created lrp_config_colab.yaml with corrected structure
merge_method: lrp

base_model:
  model: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

parameters:
  density: 0.7
  use_lrp: true

models:
  - model:
      model: "models/tinyllama-global"
    parameters:
      weight: 1.0
      lrp_scores:
        value: "./models/lrp-global/lrp_scores"
  - model:
      model: "models/tinyllama-local"
    parameters:
      weight: 1.0
      lrp_scores:
        value: "./models/lrp-local/lrp_scores"


In [ ]:
# Configuration for LOCAL model
# Using the same synthetic dataset for task-specific training
# In a real scenario, you might have a different dataset for the local model

LOCAL_OUTPUT = "models/tinyllama-local" # Corrected to local path
LOCAL_DATASET = "datasets/synthetic/train.csv"  # Can be same or different dataset

print(f"Training LOCAL model:")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {LOCAL_DATASET}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max samples: {MAX_SAMPLES}")
print(f"  Output: {LOCAL_OUTPUT}")
print(f"\nNote: Both models currently use the same dataset.")
print("For a true LRP-Merge experiment, train on different datasets:")
print("  - GLOBAL: General knowledge/tasks")
print("  - LOCAL: Specific task/domain")

Training LOCAL model:
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Dataset: datasets/synthetic/train.csv
  Epochs: 3
  Batch size: 4
  Max samples: 1000
  Output: models/tinyllama-local

Note: Both models currently use the same dataset.
For a true LRP-Merge experiment, train on different datasets:
  - GLOBAL: General knowledge/tasks
  - LOCAL: Specific task/domain


In [18]:
import os

# Using the verified LRP_PATH from previous steps
LRP_PATH = "/content/drive/MyDrive/LRP Merge method"

def make_script_portable(filename):
    # Build the full path to the script in Drive
    full_path = os.path.join(LRP_PATH, filename)

    if not os.path.exists(full_path):
        print(f"\u2717 {filename} not found at {full_path}")
        return

    with open(full_path, 'r') as f:
        content = f.read()

    # Replace hardcoded Drive paths with generic relative ones for GitHub
    # This ensures others can run it in their local folders
    old_drive_root = '/content/drive/MyDrive/LRP Merge method'
    new_content = content.replace(old_drive_root + '/models', './models')
    new_content = new_content.replace(old_drive_root + '/datasets', './datasets')
    new_content = new_content.replace(old_drive_root, '.')

    if new_content != content:
        with open(full_path, 'w') as f:
            f.write(new_content)
        print(f"\u2705 {filename} (in Drive) has been updated for GitHub portability.")
    else:
        print(f"- {filename} is already portable or clean.")

# Execute on your core scripts
scripts_to_fix = ["lrp_merge_pipeline.py", "finetune_fakenews.py", "lrp_computer.py"]
for script in scripts_to_fix:
    make_script_portable(script)

✅ lrp_merge_pipeline.py (in Drive) has been updated for GitHub portability.
- finetune_fakenews.py is already portable or clean.
- lrp_computer.py is already portable or clean.


## Download the base model

In [9]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer

LRP_PATH = "/content/drive/MyDrive/LRP Merge method" # Ensure LRP_PATH is defined
base_model_hf_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
local_base_model_path = os.path.join(LRP_PATH, "base_model")

# Ensure the directory exists
os.makedirs(local_base_model_path, exist_ok=True)

print(f"Loading base model from Hugging Face: {base_model_hf_id}")
model = AutoModelForCausalLM.from_pretrained(base_model_hf_id)
tokenizer = AutoTokenizer.from_pretrained(base_model_hf_id)

print(f"Saving base model to local path: {local_base_model_path}")
model.save_pretrained(local_base_model_path)
tokenizer.save_pretrained(local_base_model_path)

print("Base model and tokenizer saved to Drive successfully!")

Loading base model from Hugging Face: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Saving base model to local path: /content/drive/MyDrive/LRP Merge method/base_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Base model and tokenizer saved to Drive successfully!


##Merging the Global and Local model before using LRP method

In [10]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Ensure LRP_PATH is defined for this cell's scope
LRP_PATH = "/content/drive/MyDrive/LRP Merge method"

# Define full paths for base and adapter models
LOCAL_BASE_MODEL_PATH = os.path.join(LRP_PATH, "base_model")
GLOBAL_ADAPTER_PATH = os.path.join(LRP_PATH, "models", "tinyllama-global")
LOCAL_ADAPTER_PATH = os.path.join(LRP_PATH, "models", "tinyllama-local")

GLOBAL_FULL_MODEL_OUTPUT = os.path.join(LRP_PATH, "models", "tinyllama-global-full")
LOCAL_FULL_MODEL_OUTPUT = os.path.join(LRP_PATH, "models", "tinyllama-local-full")

# Ensure output directories exist
os.makedirs(GLOBAL_FULL_MODEL_OUTPUT, exist_ok=True)
os.makedirs(LOCAL_FULL_MODEL_OUTPUT, exist_ok=True)

print(f"Loading base model from: {LOCAL_BASE_MODEL_PATH}")
base = AutoModelForCausalLM.from_pretrained(LOCAL_BASE_MODEL_PATH)

print(f"Loading global adapter from: {GLOBAL_ADAPTER_PATH}")
global_model = PeftModel.from_pretrained(base, GLOBAL_ADAPTER_PATH)
print("Merging global model and unloading...")
global_model = global_model.merge_and_unload()

print(f"Saving merged global model to: {GLOBAL_FULL_MODEL_OUTPUT}")
global_model.save_pretrained(GLOBAL_FULL_MODEL_OUTPUT)

print("Reloading base model for local adapter merging...")
# Reload base again (important for fresh state if base weights are modified during merge)
base = AutoModelForCausalLM.from_pretrained(LOCAL_BASE_MODEL_PATH)

print(f"Loading local adapter from: {LOCAL_ADAPTER_PATH}")
local_model = PeftModel.from_pretrained(base, LOCAL_ADAPTER_PATH)
print("Merging local model and unloading...")
local_model = local_model.merge_and_unload()

print(f"Saving merged local model to: {LOCAL_FULL_MODEL_OUTPUT}")
local_model.save_pretrained(LOCAL_FULL_MODEL_OUTPUT)

print("✓ Global and local models merged and saved successfully!")

Loading base model from: /content/drive/MyDrive/LRP Merge method/base_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading global adapter from: /content/drive/MyDrive/LRP Merge method/models/tinyllama-global


Merging global model and unloading...
Saving merged global model to: /content/drive/MyDrive/LRP Merge method/models/tinyllama-global-full


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Reloading base model for local adapter merging...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading local adapter from: /content/drive/MyDrive/LRP Merge method/models/tinyllama-local
Merging local model and unloading...
Saving merged local model to: /content/drive/MyDrive/LRP Merge method/models/tinyllama-local-full


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Global and local models merged and saved successfully!


##Download the custom mergekit repo before applying LRP method

In [ ]:
import os

mergekit_repo_path = os.path.join(LRP_PATH, "mergekit_repo")

if os.path.exists(mergekit_repo_path) and os.path.isdir(mergekit_repo_path):
    print(f"Found custom mergekit_repo at: {mergekit_repo_path}")
    print("Installing custom mergekit_repo in editable mode...")
    !pip install -e "{mergekit_repo_path}"
    print("✓ Custom mergekit_repo installed.")
else:
    print(f"✗ Custom mergekit_repo not found at: {mergekit_repo_path}")
    print("Please ensure 'mergekit_repo' directory containing your LRP implementation is present in your Google Drive at: {LRP_PATH}/mergekit_repo")
    #Exit if custom mergekit_repo is not found, as the LRP method will not work.
    import sys
    sys.exit("Error: Custom mergekit_repo not found. Please place it in your Google Drive as specified.")

# Re-check available merge methods after installation
print("Checking available merge methods in mergekit after custom installation...")
print("Note: A full check of merge methods might cause a SystemExit if the custom method isn't fully integrated yet. Proceeding with script fixes.")

Found custom mergekit_repo at: /content/drive/MyDrive/LRP Merge method/mergekit_repo
Installing custom mergekit_repo in editable mode...
Obtaining file:///content/drive/MyDrive/LRP%20Merge%20method/mergekit_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mergekit (pyproject.toml) ... done
  Created wheel for mergekit: filename=mergekit-0.1.4-0.editable-py3-none-any.whl size=13739 sha256=f31b6550c62ceb245bb585adcad39ca2e971e5417c687335412eb746279d62c8
  Stored in directory: /tmp/pip-ephem-wheel-cache-vpbngun1/wheels/02/0c/3b/52585b782f267fb895caf8fddcee3dfacd3df6743ef269b5a3
Successfully built mergekit
  Attempting uninstall: mergekit
    Found existing installation: mergekit 0.1.4
    Uninstalling mergekit-0.1.4:
      Successfully uninstalled mergekit-0.1.4
✓ Custom mergekit_repo installed.
Checking

## Test the Merged Model

In [76]:

# It uses CPU mode to ensure stability across different environment configurations
import os
import shutil
import subprocess

def run_final_merge():
    config_path = "lrp_config.yaml"
    output_dir = os.path.join(LRP_PATH, "models/merged-model")
    os.makedirs(output_dir, exist_ok=True)

    cmd = [
        "mergekit-yaml",
        config_path,
        output_dir,
        "--copy-tokenizer",
        "--allow-crimes",
        "--lazy-unpickle"
    ]

    print(f"Executing: {' '.join(cmd)}")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout: print(line, end="")
    process.wait()
    if process.returncode == 0: print("\n✅ LRP MERGE SUCCESSFUL")

run_final_merge()


Step 3: Running merge (LRP-Merge / CPU Mode)...

Running command: /usr/local/bin/mergekit-yaml lrp_config.yaml /content/drive/MyDrive/LRP Merge method/models/merged-model --copy-tokenizer --allow-crimes --lazy-unpickle

Warmup loader cache:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 8818.51it/s]

Warmup loader cache: 100%|██████████| 3/3 [00:00<00:00, 20.23it/s]

Executing graph: 100%|██████████| 1208/1208 [04:20<00:00,  4.63it/s]

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 15592.21it/s]

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 12515.64it/s]

✅ MERGE SUCCESSFUL


## Testing the final merged model

In [77]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

# Path to your freshly merged model
MERGED_MODEL_PATH = os.path.join(LRP_PATH, "models/merged-model")

print(f"Loading merged model from: {MERGED_MODEL_PATH}")

try:
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # Prepare a test prompt
    prompt = "### Instruction: Classify this news as REAL or FAKE.\n\n### Input: Scientists discover new planet using James Webb telescope.\n\n### Response:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    print("\nGenerating response...")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1)

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("="*50)
    print("Test Result:")
    print(result)
    print("="*50)

except Exception as e:
    print(f"Error testing model: {e}")
    print("Make sure the merge process finished successfully and the files exist in Drive.")

Loading merged model from: /content/drive/MyDrive/LRP Merge method/models/merged-model


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Generating response...
Test Result:
### Instruction: Classify this news as REAL or FAKE.

### Input: Scientists discover new planet using James Webb telescope.

### Response: REAL

### Input: Biden


In [ ]:
!pip install -q -U mergekit

In [4]:
import os
from google.colab import drive

# Check if drive is still mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("Drive not detected. Re-mounting...")
    drive.mount('/content/drive', force_remount=True)

# List contents of MyDrive to verify the folder name
print("Contents of MyDrive:")
!ls -d /content/drive/MyDrive/LRP*

# Reset LRP_PATH based on verified name
# (Sometimes spaces or casing in folder names cause issues)
LRP_PATH = "/content/drive/MyDrive/LRP Merge method"
print(f"\nLRP_PATH set to: {LRP_PATH}")

Drive not detected. Re-mounting...
Mounted at /content/drive
Contents of MyDrive:
'/content/drive/MyDrive/LRP Merge method'

LRP_PATH set to: /content/drive/MyDrive/LRP Merge method


## Evaluation of the model

In [80]:
# Test Evaluation
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

path = os.path.join(LRP_PATH, "models/merged-model")
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32, device_map="auto")

test_data = [
    {"text": "NASA announces new space mission to Mars.", "label": "REAL"},
    {"text": "Aliens found living on the moon.", "label": "FAKE"}
]

for item in test_data:
    inputs = tokenizer(f"### Input: {item['text']}\n### Response:", return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=5)
    print(f"Text: {item['text']} | Prediction: {tokenizer.decode(out[0], skip_special_tokens=True).split('### Response:')[-1].strip()}")

Loading model for evaluation: /content/drive/MyDrive/LRP Merge method/models/merged-model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Starting Evaluation...

                                       Text Expected Predicted Match
NASA announces new space mission to Mars...     REAL      REAL     ✅
        Aliens found living on the moon....     FAKE      REAL     ❌
 Government increases education funding....     REAL      REAL     ✅
      Secret cure for cancer discovered....     FAKE      FAKE     ✅
Final Accuracy: 75.00%


## Save Results to Google Drive

Ensure all trained models are saved back to your Google Drive.

In [ ]:
import shutil

# List what we have
!ls -lh models/

# If models are in /content (not Drive), copy them
if not LRP_PATH.startswith("/content/drive"):
    DRIVE_PATH = "/content/drive/MyDrive/LRP merge method/models"
    os.makedirs(DRIVE_PATH, exist_ok=True)

    # Copy trained models
    for model_dir in ["tinyllama-global", "tinyllama-local", "merged-model"]:
        src = f"models/{model_dir}"
        dst = f"{DRIVE_PATH}/{model_dir}"
        if os.path.exists(src):
            print(f"Copying {src} to Drive...")
            shutil.copytree(src, dst, dirs_exist_ok=True)

    print(f"\n✓ Models copied to: {DRIVE_PATH}")
else:
    print("✓ Models already saved to Google Drive")

## Download Trained Models (Optional)

Download the trained models to your local machine.

In [ ]:
# Create a ZIP file of the models
!zip -r trained_models.zip models/

# Download the ZIP
from google.colab import files
files.download("trained_models.zip")

print("Download started!")